# Fraud Detection with Temporal Feature Engineering

This notebook investigates whether engineered temporal transaction features improve machine-learning fraud detection under severe class imbalance.

The original project compared **Logistic Regression**, **Random Forest**, and **XGBoost** with and without engineered temporal features. This cleaned notebook reorganizes the original analysis into a readable, reproducible structure while preserving the core experimental setup and conclusions.

> **Main finding:** temporal features did not substantially improve minority-class fraud detection. Static/account-level attributes remained more influential, and ROC-AUC stayed close to 0.50 across the tested models.

## 1. Setup

The notebook expects the dataset at:

`data/Bank_Transaction_Fraud_Detection.csv`

The dataset itself is intentionally not included in this repository until its redistribution license is confirmed.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier

RANDOM_STATE = 42
DATA_PATH = Path("data/Bank_Transaction_Fraud_Detection.csv")

pd.set_option("display.max_columns", 100)

## 2. Load and inspect the data

The original dataset contains transaction-level records including transaction amount, date/time, customer/account information, and a binary fraud label.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)

print("Shape:", df_raw.shape)
print("\nFraud distribution:")
print(df_raw["Is_Fraud"].value_counts(dropna=False))
print("\nFraud rate:")
print(df_raw["Is_Fraud"].mean())

df_raw.head()

## 3. Basic exploratory analysis

The target is highly imbalanced, with fraudulent transactions representing only a small fraction of all observations.

In [ ]:
fraud_share = df_raw["Is_Fraud"].value_counts(normalize=True).sort_index()

ax = fraud_share.plot(kind="bar", figsize=(6, 4))
ax.set_title("Fraud Class Distribution")
ax.set_xlabel("Is_Fraud")
ax.set_ylabel("Share of transactions")
plt.tight_layout()
plt.show()

## 4. Temporal feature engineering

The original project engineered behavioral features intended to capture short-term and time-dependent transaction patterns:

- number of transactions in the previous 1-day rolling window
- 1-day and 7-day rolling transaction amount means
- account balance change rate
- time since the previous transaction
- transaction amount z-score within customer
- hour of day
- nighttime indicator
- weekday
- daily transaction count

The code below preserves the feature definitions from the original analysis.

In [ ]:
def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["Transaction_DateTime"] = pd.to_datetime(
        df["Transaction_Date"] + " " + df["Transaction_Time"],
        format="%d-%m-%Y %H:%M:%S",
        errors="coerce",
    )
    df = df.sort_values(["Customer_ID", "Transaction_DateTime"])

    feature_cols = [
        "txns_last_1d",
        "mean_amt_1d",
        "mean_amt_7d",
        "balance_change_rate",
        "time_since_last",
        "amt_zscore",
    ]
    for col in feature_cols:
        df[col] = np.nan

    for _, group in df.groupby("Customer_ID"):
        group = group.sort_values("Transaction_DateTime")
        idx = group.index
        group_roll = group.set_index("Transaction_DateTime")

        df.loc[idx, "txns_last_1d"] = (
            group_roll["Account_Balance"].rolling("1d").count().values
        )
        df.loc[idx, "mean_amt_1d"] = (
            group_roll["Transaction_Amount"].rolling("1d").mean().values
        )
        df.loc[idx, "mean_amt_7d"] = (
            group_roll["Transaction_Amount"].rolling("7d").mean().values
        )

        prev_balance = group["Account_Balance"].shift(1)
        balance_change = (
            (group["Account_Balance"] - prev_balance)
            / prev_balance.replace(0, np.nan)
        )
        df.loc[idx, "balance_change_rate"] = balance_change.fillna(0).values

        prev_time = group["Transaction_DateTime"].shift(1)
        time_diff = (
            group["Transaction_DateTime"] - prev_time
        ).dt.total_seconds()
        df.loc[idx, "time_since_last"] = (
            time_diff.fillna(time_diff.median()).values
        )

        mean_amt = group["Transaction_Amount"].mean()
        std_amt = group["Transaction_Amount"].std()
        df.loc[idx, "amt_zscore"] = (
            (group["Transaction_Amount"] - mean_amt) / (std_amt + 1e-6)
        ).values

    df["hour"] = df["Transaction_DateTime"].dt.hour
    df["is_night"] = df["hour"].between(0, 6).astype(int)
    df["weekday"] = df["Transaction_DateTime"].dt.weekday
    df["daily_txn_count"] = (
        df.groupby(df["Transaction_DateTime"].dt.date)["Transaction_ID"]
        .transform("count")
    )

    df["txns_last_1d"] = df["txns_last_1d"].fillna(0)
    df["mean_amt_1d"] = df["mean_amt_1d"].fillna(df["Transaction_Amount"])
    df["mean_amt_7d"] = df["mean_amt_7d"].fillna(df["Transaction_Amount"])

    return df


df_temporal = add_temporal_features(df_raw)

temporal_features = [
    "txns_last_1d",
    "mean_amt_1d",
    "mean_amt_7d",
    "balance_change_rate",
    "time_since_last",
    "amt_zscore",
    "hour",
    "is_night",
    "weekday",
    "daily_txn_count",
]

df_temporal[temporal_features].head()

## 5. Preprocessing

The original analysis encoded categorical columns using `LabelEncoder`, filled remaining missing values with zero, and used a stratified 70/30 train-test split.

**Important:** this section intentionally reflects the original project design. A stronger production/research pipeline would fit preprocessing only on the training split and would use an encoding strategy designed for nominal categorical variables.

In [ ]:
def encode_categorical_columns(df: pd.DataFrame) -> pd.DataFrame:
    encoded = df.copy()
    encoder = LabelEncoder()

    for col in encoded.select_dtypes(include="object").columns:
        encoded[col] = encoder.fit_transform(encoded[col].astype(str))

    return encoded


df_model = encode_categorical_columns(df_temporal)
df_model = df_model.fillna(0)

X_temporal = df_model.drop(columns=["Is_Fraud", "Transaction_DateTime"])
y = df_model["Is_Fraud"]

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_temporal,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train_t.shape)
print("Test shape:", X_test_t.shape)

## 6. Class imbalance and SMOTE

The original analysis applied **SMOTE only to the training data after the train-test split**. This is preferable to oversampling the full dataset before splitting, because the test set remains untouched.

However, there are still limitations:

1. SMOTE creates synthetic points using nearest neighbors. When nominal categorical variables have first been integer-encoded, interpolating between those integer codes does not have a natural categorical interpretation.
2. Temporal data can require time-aware validation. A random train-test split may allow future observations to influence an experiment intended to model past-to-future behavior.
3. Several temporal features in the original code are constructed before the train-test split, so a stricter follow-up analysis should engineer features in a leakage-aware pipeline.

For this portfolio notebook, SMOTE is retained to document the original experiment rather than silently changing the methodology.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_t, y_train_t
)

print("Original training distribution:")
print(y_train_t.value_counts())

print("\nAfter SMOTE:")
print(y_train_resampled.value_counts())

## 7. Model definitions

Three supervised learning models were compared:

- Logistic Regression
- Random Forest
- XGBoost

In [ ]:
def make_models():
    return {
        "XGBoost": XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            max_depth=12,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
        ),
        "Logistic Regression": LogisticRegression(
            max_iter=2000,
            solver="liblinear",
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
    }

## 8. Evaluation helper

Because fraud is a minority class, accuracy alone is not sufficient. The experiment reports precision, recall, F1-score, and ROC-AUC alongside accuracy.

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, scale=False):
    if scale:
        scaler = StandardScaler()
        X_train_fit = scaler.fit_transform(X_train)
        X_test_fit = scaler.transform(X_test)
    else:
        X_train_fit = X_train
        X_test_fit = X_test

    model.fit(X_train_fit, y_train)

    y_pred = model.predict(X_test_fit)
    y_prob = model.predict_proba(X_test_fit)[:, 1]

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
    }

## 9. Temporal-feature models

This cell trains the three models using the engineered temporal feature set.

> Running all models on the full dataset may take several minutes depending on hardware.

In [ ]:
temporal_results = {}

for name, model in make_models().items():
    temporal_results[name] = evaluate_model(
        model,
        X_train_resampled,
        y_train_resampled,
        X_test_t,
        y_test_t,
        scale=(name == "Logistic Regression"),
    )

pd.DataFrame(temporal_results).T

## 10. Static / non-engineered baseline

The original project compared the temporal-feature models with models trained without the engineered temporal variables.

The original code's baseline still retained some raw date/time-related columns after categorical encoding. Therefore, this notebook describes it more precisely as **"without engineered temporal features"** rather than claiming it is a perfectly time-free baseline.

In [ ]:
engineered_temporal_cols = temporal_features + ["Transaction_DateTime"]

baseline_df = df_temporal.drop(
    columns=engineered_temporal_cols,
    errors="ignore",
)
baseline_df = encode_categorical_columns(baseline_df).fillna(0)

X_baseline = baseline_df.drop(columns=["Is_Fraud"])
y_baseline = baseline_df["Is_Fraud"]

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_baseline,
    y_baseline,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y_baseline,
)

X_train_b_resampled, y_train_b_resampled = SMOTE(
    random_state=RANDOM_STATE
).fit_resample(X_train_b, y_train_b)

baseline_results = {}

for name, model in make_models().items():
    baseline_results[name] = evaluate_model(
        model,
        X_train_b_resampled,
        y_train_b_resampled,
        X_test_b,
        y_test_b,
        scale=(name == "Logistic Regression"),
    )

pd.DataFrame(baseline_results).T

## 11. Results reported in the original study

The original study reported the following held-out performance:

| Model | Engineered Temporal Features | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---:|---:|---:|---:|---:|---:|
| XGBoost | Yes | 0.9500 | 0.0400 | 0.0000 | 0.0000 | 0.4936 |
| Random Forest | Yes | 0.7900 | 0.0500 | 0.1700 | 0.0800 | 0.4956 |
| Logistic Regression | Yes | 0.7600 | 0.0500 | 0.2100 | 0.0800 | 0.5005 |
| XGBoost | No | 0.7896 | 0.0508 | 0.1794 | 0.0792 | 0.5010 |
| Random Forest | No | 0.7404 | 0.0507 | 0.2340 | 0.0833 | 0.5015 |
| Logistic Regression | No | 0.6371 | 0.0521 | 0.3605 | 0.0911 | 0.5022 |

The key result is not the high accuracy of some configurations. Because the dataset is highly imbalanced, the minority-class metrics and ROC-AUC are more informative. The temporal features did **not** materially improve fraud discrimination in this experiment.

## 12. Feature importance

The original XGBoost analysis found that static/account-level variables were generally more influential than the engineered temporal variables. The code below reproduces the feature-importance view for the temporal-feature XGBoost model.

In [ ]:
xgb_model = make_models()["XGBoost"]
xgb_model.fit(X_train_resampled, y_train_resampled)

importance = (
    pd.Series(
        xgb_model.feature_importances_,
        index=X_train_resampled.columns,
        name="importance",
    )
    .sort_values(ascending=False)
    .head(20)
)

ax = importance.sort_values().plot(kind="barh", figsize=(8, 7))
ax.set_title("Top XGBoost Feature Importances")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

## 13. Key findings and limitations

### Key findings

- The dataset is strongly imbalanced, with fraud representing roughly 5% of transactions.
- Engineered temporal variables captured behavioral information, but they did not substantially improve minority-class fraud detection.
- In the reported experiment, ROC-AUC remained approximately 0.49–0.50 across models.
- Static/account-level features were more influential than the engineered temporal features.

### Methodological limitations

This project was an exploratory study, and the original pipeline has limitations worth addressing in follow-up work:

- temporal features were engineered before the random train-test split;
- random splitting is not ideal for strictly temporal prediction tasks;
- label encoding imposes arbitrary numeric order on nominal categories;
- SMOTE on integer-encoded categorical variables can create synthetic values without a clear categorical interpretation;
- the "no temporal features" comparison in the original code retained raw date/time-related fields after encoding.

A stronger follow-up would use a chronological split, train-only preprocessing, appropriate categorical encoding, and imbalance strategies compatible with mixed feature types.